<a href="https://colab.research.google.com/github/trulydeeprogrmr29/PyTorch_Project/blob/main/pytorch_lstm_next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install nltk

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [4]:
document = """The year 1947 is very important in Indian history.
India became free from British rule on 15 August 1947.
Before this, the British ruled India for many years.
Many brave people fought for India's freedom.
They made great sacrifices for the country.
Mahatma Gandhi was one of the greatest freedom fighters.
He believed in peace and non-violence.
Jawaharlal Nehru became the first Prime Minister of India.
Lord Mountbatten was the last British Viceroy.
On Independence Day, people celebrated with great joy.
The Indian national flag was hoisted everywhere.
People sang patriotic songs.
They took part in parades and public meetings.
Jawaharlal Nehru gave his famous "Tryst with Destiny" speech.
Everyone hoped for a better future.
But 1947 also brought many problems.
India was divided into India and Pakistan.
This division is called the Partition of India.
Many families had to leave their homes.
Millions of people moved to new places.
Many people lost their lives in violence.
Many children became homeless.
Refugee camps were built to help people.
Mahatma Gandhi worked to bring peace.
He asked people to live together in harmony.
The new government faced many challenges.
There was a shortage of food.
Many people were poor.
Most people worked as farmers.
Schools were not available in every village.
Hospitals were also few.
The government wanted to improve education.
It also wanted to improve healthcare.
Roads and railways needed more development.
The leaders worked hard to build a strong nation.
The Constituent Assembly started preparing the Constitution.
Dr. B. R. Ambedkar played an important role.
India decided to become a democratic country.
Every citizen was given equal importance.
Many princely states joined India.
Sardar Vallabhbhai Patel helped unite these states.
This made India stronger.
People worked together to rebuild the country.
They wanted peace and progress.
Young people dreamed of a bright future.
Women also helped in building the nation.
The freedom fighters were respected by everyone.
Their sacrifices will never be forgotten.
India started a new journey as an independent nation.
The year 1947 will always be remembered with pride by every Indian.  Mahatma Gandhi was born on 2 October 1869.
His full name was Mohandas Karamchand Gandhi.
He was born in Porbandar, Gujarat.
His father's name was Karamchand Gandhi.
His mother's name was Putlibai Gandhi.
Gandhi was known as the Father of the Nation in India.
He studied law in London.
He became a lawyer after completing his studies.
He worked in South Africa for many years.
Gandhi fought against racial discrimination in South Africa.
He believed in truth.
He believed in non-violence.
He called truth Satya.
He called non-violence Ahimsa.
Gandhi inspired millions of people.
He returned to India in 1915.
He joined the Indian freedom movement.
He led many peaceful protests.
He encouraged people to protest without violence.
Gandhi wore simple hand-spun cotton clothes.
He promoted the use of Khadi.
He encouraged people to spin cotton on the Charkha.
He wanted India to become self-reliant.
He started the Non-Cooperation Movement.
He started the Civil Disobedience Movement.
He led the famous Dandi Salt March in 1930.
The Salt March protested the British salt tax.
Gandhi was arrested many times.
He spent several years in prison.
He often observed fasts for peace.
He wanted people of all religions to live together.
He spoke against untouchability.
He called the poor and oppressed people Harijans.
He believed in equality.
Gandhi lived a simple life.
He believed that actions are more important than words.
He inspired many leaders around the world.
His ideas influenced Martin Luther King Jr.
His ideas also influenced Nelson Mandela.
Gandhi loved reading and writing.
He wrote many letters and articles.
His autobiography is called "The Story of My Experiments with Truth."
Gandhi believed in honesty.
He believed in discipline and hard work.
He wanted every child to receive education.
He worked for peace until the end of his life.
Gandhi was assassinated on 30 January 1948.
His death shocked the whole nation.
Every year, 2 October is celebrated as Gandhi Jayanti in India.
Mahatma Gandhi is remembered as one of the greatest leaders in history.
"""


In [5]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
# tokenize
tokens = word_tokenize(document.lower())

In [7]:
# build vocab
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<unk>': 0,
 'the': 1,
 'year': 2,
 '1947': 3,
 'is': 4,
 'very': 5,
 'important': 6,
 'in': 7,
 'indian': 8,
 'history': 9,
 '.': 10,
 'india': 11,
 'became': 12,
 'free': 13,
 'from': 14,
 'british': 15,
 'rule': 16,
 'on': 17,
 '15': 18,
 'august': 19,
 '1947.': 20,
 'before': 21,
 'this': 22,
 ',': 23,
 'ruled': 24,
 'for': 25,
 'many': 26,
 'years': 27,
 'brave': 28,
 'people': 29,
 'fought': 30,
 "'s": 31,
 'freedom': 32,
 'they': 33,
 'made': 34,
 'great': 35,
 'sacrifices': 36,
 'country': 37,
 'mahatma': 38,
 'gandhi': 39,
 'was': 40,
 'one': 41,
 'of': 42,
 'greatest': 43,
 'fighters': 44,
 'he': 45,
 'believed': 46,
 'peace': 47,
 'and': 48,
 'non-violence': 49,
 'jawaharlal': 50,
 'nehru': 51,
 'first': 52,
 'prime': 53,
 'minister': 54,
 'lord': 55,
 'mountbatten': 56,
 'last': 57,
 'viceroy': 58,
 'independence': 59,
 'day': 60,
 'celebrated': 61,
 'with': 62,
 'joy': 63,
 'national': 64,
 'flag': 65,
 'hoisted': 66,
 'everywhere': 67,
 'sang': 68,
 'patriotic': 69,
 'so

In [8]:
len(vocab)

320

In [9]:
input_sentences = document.split('\n')

In [10]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [11]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))


In [12]:
len(input_numerical_sentences)

100

In [13]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [14]:
len(training_sequence)

692

In [15]:
training_sequence[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]

In [16]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

22

In [17]:
training_sequence[0]

[1, 2]

In [18]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [19]:
len(padded_training_sequence[10])

22

In [20]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [21]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0,   0,  ...,  43, 152,   7],
        [  0,   0,   0,  ..., 152,   7,   9],
        [  0,   0,   0,  ...,   7,   9,  10]])

In [22]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [23]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   0,  ...,   1,  43, 152],
        [  0,   0,   0,  ...,  43, 152,   7],
        [  0,   0,   0,  ..., 152,   7,   9]])

In [24]:
y

tensor([  2,   3,   4,   5,   6,   7,   8,   9,  10,  12,  13,  14,  15,  16,
         17,  18,  19,   3,  10,  22,  23,   1,  15,  24,  11,  25,  26,  27,
         10,  28,  29,  30,  25,  11,  31,  32,  10,  34,  35,  36,  25,   1,
         37,  10,  39,  40,  41,  42,   1,  43,  32,  44,  10,  46,   7,  47,
         48,  49,  10,  51,  12,   1,  52,  53,  54,  42,  11,  10,  56,  40,
          1,  57,  15,  58,  10,  59,  60,  23,  29,  61,  62,  35,  63,  10,
          8,  64,  65,  40,  66,  67,  10,  68,  69,  70,  10,  71,  72,   7,
         73,  48,  74,  75,  10,  51,  76,  77,  78,  79,  80,  62,  81,  82,
         83,  10,  85,  25,  86,  87,  88,  10,   3,  90,  91,  26,  92,  10,
         40,  93,  94,  11,  48,  95,  10,  96,   4,  97,   1,  98,  42,  11,
         10,  99, 100, 101, 102, 103, 104,  10,  42,  29, 106, 101, 107, 108,
         10,  29, 109, 103, 110,   7, 111,  10, 112,  12, 113,  10, 115, 116,
        117, 101, 118,  29,  10,  39, 119, 101, 120,  47,  10, 1

In [25]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [26]:
dataset = CustomDataset(X,y)

In [27]:
len(dataset)

692

In [28]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [29]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [30]:
model = LSTMModel(len(vocab))

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [32]:
model.to(device)

LSTMModel(
  (embedding): Embedding(320, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=320, bias=True)
)

In [33]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [34]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 124.5130
Epoch: 2, Loss: 110.6821
Epoch: 3, Loss: 102.9831
Epoch: 4, Loss: 97.4500
Epoch: 5, Loss: 91.8651
Epoch: 6, Loss: 85.6191
Epoch: 7, Loss: 79.8176
Epoch: 8, Loss: 73.9114
Epoch: 9, Loss: 67.9664
Epoch: 10, Loss: 62.6267
Epoch: 11, Loss: 57.1849
Epoch: 12, Loss: 52.3207
Epoch: 13, Loss: 47.5695
Epoch: 14, Loss: 43.0313
Epoch: 15, Loss: 38.7819
Epoch: 16, Loss: 35.0012
Epoch: 17, Loss: 31.5127
Epoch: 18, Loss: 28.2415
Epoch: 19, Loss: 25.7762
Epoch: 20, Loss: 23.2389
Epoch: 21, Loss: 21.2049
Epoch: 22, Loss: 19.5953
Epoch: 23, Loss: 17.9779
Epoch: 24, Loss: 16.5705
Epoch: 25, Loss: 15.5855
Epoch: 26, Loss: 14.7667
Epoch: 27, Loss: 13.7123
Epoch: 28, Loss: 13.1554
Epoch: 29, Loss: 12.3309
Epoch: 30, Loss: 11.8906
Epoch: 31, Loss: 11.4329
Epoch: 32, Loss: 11.1180
Epoch: 33, Loss: 10.6291
Epoch: 34, Loss: 10.2036
Epoch: 35, Loss: 10.0600
Epoch: 36, Loss: 9.8473
Epoch: 37, Loss: 9.6312
Epoch: 38, Loss: 9.3267
Epoch: 39, Loss: 9.1457
Epoch: 40, Loss: 8.9794
Epoch: 41, 

In [35]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]



In [36]:
prediction(model, vocab, "Gandhi was assassinated on 30 January 1948")

'Gandhi was assassinated on 30 January 1948 .'

In [40]:
import time

num_tokens = 10
input_text = "every year"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


every year ,
every year , 2
every year , 2 october
every year , 2 october is
every year , 2 october is celebrated
every year , 2 october is celebrated as
every year , 2 october is celebrated as gandhi
every year , 2 october is celebrated as gandhi jayanti
every year , 2 october is celebrated as gandhi jayanti in
every year , 2 october is celebrated as gandhi jayanti in india


In [38]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [41]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")


Model Accuracy: 88.87%
